# Puente de Keras para PuntoBot

Práctica de la sesión 22 adaptada a cuatro intenciones de texto. TF-IDF convierte cada consulta en características numéricas; no se usa la etiqueta, el producto ni el rol como entrada.

En Colab, autoriza un secreto llamado `GITHUB_TOKEN` con lectura de este repositorio privado. En tu PC, instala `requirements-keras.txt` y abre el notebook desde el repositorio. Nunca pegues el token en una celda. Los datos de demostración son sintéticos; `test.csv` se mantiene sellado.

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, io, urllib.request, subprocess

# En Colab: abre el notebook desde GitHub y autoriza el secreto GITHUB_TOKEN.
# El token se usa solo en una cabecera HTTP y nunca se imprime ni se guarda.
if Path('/content').exists() and not Path('config.yaml').exists():
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    request = urllib.request.Request(
        'https://api.github.com/repos/rebirth18-09/puntobot-ia/zipball/main',
        headers={'Authorization': 'Bearer ' + token, 'Accept': 'application/vnd.github+json'})
    with urllib.request.urlopen(request) as response:
        contenido = response.read()
    del token, request
    destino = Path('/content/puntobot-ia')
    destino.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(contenido)) as archivo:
        for item in archivo.infolist():
            target = (destino / item.filename).resolve()
            if not target.is_relative_to(destino.resolve()):
                raise ValueError('Ruta inválida en el ZIP')
        archivo.extractall(destino)
    os.chdir(next(p for p in destino.iterdir() if (p / 'config.yaml').exists()))
RAIZ = Path.cwd()
if not (RAIZ / 'config.yaml').exists() and (RAIZ.parent / 'config.yaml').exists():
    RAIZ = RAIZ.parent
assert (RAIZ / 'config.yaml').exists(), 'Ejecuta el notebook desde la carpeta del repositorio'
os.chdir(RAIZ)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
if Path('/content').exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-keras.txt'], check=True)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
import tensorflow as tf
import keras
import numpy as np
import yaml
try:
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.config.threading.set_intra_op_parallelism_threads(1)
except RuntimeError:
    pass
config = yaml.safe_load((RAIZ / 'config.yaml').read_text('utf-8'))
keras.utils.set_random_seed(config['entrenamiento']['semilla'])
print('TensorFlow', tf.__version__, 'Keras', keras.__version__)
print('GPU', tf.config.list_physical_devices('GPU'))

## Carga y transformación sin fuga de datos

In [ ]:
from src.data.division import cargar_particiones
from src.models.modelo import crear_vectorizador
from src.models.pesos import calcular_pesos
train, val, auditoria = cargar_particiones(config, RAIZ)
texto, etiqueta = config['datos']['columna_texto'], config['datos']['columna_clase']
vectorizador = crear_vectorizador(config)
# El único fit del vocabulario y de IDF usa entrenamiento.
X_train = vectorizador.fit_transform(train[texto]).toarray().astype('float32')
X_val = vectorizador.transform(val[texto]).toarray().astype('float32')
mapa = {c: i for i, c in enumerate(config['clases'])}
y_train = train[etiqueta].map(mapa).to_numpy(dtype='int32')
y_val = val[etiqueta].map(mapa).to_numpy(dtype='int32')
pesos = calcular_pesos(train[etiqueta].tolist(), config['clases'])
pesos_keras = {mapa[c]: p for c, p in pesos.items()}
print('Formas', X_train.shape, X_val.shape)
print('Prueba sellada:', auditoria['filas_prueba'], 'filas')
print('Pesos solo train:', pesos)

## Definir y compilar

La API funcional conecta entrada, Dense 16 ReLU, Dense 8 ReLU y Dense 4 softmax. Cuatro clases excluyentes requieren `sparse_categorical_crossentropy` con etiquetas 0 a 3.

In [ ]:
from src.models.puente_keras import crear_modelo_puente
modelo = crear_modelo_puente(X_train.shape[1], config)
n = X_train.shape[1]
dimensiones = [n, *config['puente_keras']['neuronas_ocultas'], len(config['clases'])]
parametros_manual = sum(a*b+b for a,b in zip(dimensiones, dimensiones[1:]))
assert modelo.count_params() == parametros_manual
print('Parámetros calculados:', parametros_manual)
modelo.summary()

## Entrenar

Adam ajusta los pesos con tasa 0.001. Un lote contiene 32 ejemplos; una época recorre train. Los pesos balanceados compensan clases pequeñas. Se observa val sin ajustar su vocabulario.

In [ ]:
import math
print('Pasos por época:', math.ceil(len(y_train)/config['entrenamiento']['tamano_lote']))
historial = modelo.fit(X_train, y_train, validation_data=(X_val, y_val),
    epochs=config['entrenamiento']['epocas_puente'], batch_size=config['entrenamiento']['tamano_lote'],
    class_weight=pesos_keras, verbose=0)
print('Pérdida:', historial.history['loss'][0], '→', historial.history['loss'][-1])

In [ ]:
import matplotlib.pyplot as plt
plt.plot(historial.history['loss'], label='entrenamiento')
plt.plot(historial.history['val_loss'], label='validación')
plt.xlabel('Época'); plt.ylabel('Pérdida'); plt.legend(); plt.show()

## Evaluar y predecir

Para respetar el protocolo de PuntoBot, esta práctica usa validación y reserva prueba para la etapa 7. La evaluación aquí comprueba el ciclo de Keras y no determina el modelo final.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
resultado = modelo.evaluate(X_val, y_val, return_dict=True, verbose=0)
probabilidades = modelo.predict(X_val, verbose=0)
pred = probabilidades.argmax(axis=1)
mayoritaria = np.bincount(y_train).argmax()
base = np.full(len(y_val), mayoritaria)
resumen = {'keras': resultado, 'keras_f1_macro': float(f1_score(y_val, pred, labels=list(mapa.values()), average='macro', zero_division=0)),
    'baseline_accuracy': float(accuracy_score(y_val, base)),
    'baseline_f1_macro': float(f1_score(y_val, base, labels=list(mapa.values()), average='macro', zero_division=0))}
print(json.dumps(resumen, indent=2))
print('Cinco predicciones:', [config['clases'][i] for i in pred[:5]])
assert np.allclose(probabilidades.sum(axis=1), 1, atol=1e-5)

## Romper y reparar

Las tres comprobaciones usan objetos de prueba independientes y conservan el modelo correcto.

In [ ]:
# 1. Una característica extra provoca incompatibilidad de forma.
mal_formado = crear_modelo_puente(n + 1, config)
try:
    mal_formado.predict(X_train[:1], verbose=0)
except ValueError as error:
    print('Forma incorrecta detectada:', type(error).__name__)
else:
    raise AssertionError('La forma incorrecta no se detectó')
assert modelo.input_shape[-1] == n
# 2. Una salida lineal no garantiza probabilidades.
entrada = keras.Input(shape=(n,))
lineal = keras.layers.Dense(len(config['clases']), kernel_initializer='zeros', bias_initializer=keras.initializers.Constant(-1))(entrada)
red_lineal = keras.Model(entrada, lineal)
assert (red_lineal.predict(X_train[:1], verbose=0) < 0).any()
print('Salida sin softmax inválida; el modelo correcto usa softmax.')
# 3. Se demuestra la fuga con textos artificiales, sin usar test.
from sklearn.feature_extraction.text import TfidfVectorizer
solo_train = TfidfVectorizer().fit(['datoentrenamiento'])
con_fuga = TfidfVectorizer().fit(['datoentrenamiento','datovalidacion'])
assert 'datovalidacion' not in solo_train.vocabulary_
assert 'datovalidacion' in con_fuga.vocabulary_
print('Fuga detectada: vocabulario aprendido fuera de entrenamiento.')

## Continuidad

El puente es didáctico. El clasificador del proyecto mantiene la decisión de la etapa 4: TF-IDF más regresión logística. Revisar los resultados con otro integrante antes de la entrega académica.